# 实验一 静态网页与 API 数据采集

- 姓名：柳兆星
- 学号：112404150131
- 班级：数据科学与大数据技术软件1241班

本 Notebook 演示 QQ 表情 JSON API 采集和手机归属地 HTML 采集。

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import mimetypes
import os
import re
import requests

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36'
    )
}
print('Requests 版本：', requests.__version__)

## 任务一 QQ 表情包图片下载

In [ ]:
def download_qq_stickers(keyword='开心', count=5, output_dir='qq_stickers'):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # 1. 定义请求信息
    search_url = 'https://h5api.sginput.qq.com/wxbq/search'
    params = {'key': keyword, 'page': 1, 'num': count}

    # 2. 发送请求并获得响应
    response = requests.get(search_url, headers=HEADERS, params=params, timeout=15)
    response.raise_for_status()
    response.encoding = 'utf-8'
    items = response.json().get('data', [])[:count]

    saved_files = []
    for index, item in enumerate(items, start=1):
        image_url = item.get('indexUrl')
        if not image_url:
            continue
        image_response = requests.get(image_url, headers=HEADERS, timeout=15)
        image_response.raise_for_status()
        content_type = image_response.headers.get('Content-Type', '').split(';')[0]
        suffix = mimetypes.guess_extension(content_type) or Path(urlparse(image_url).path).suffix or '.jpg'
        if suffix == '.jpe':
            suffix = '.jpg'
        destination = output_path / f'{index:02d}_{keyword}{suffix}'

        # 3. 将响应图片存储到本地
        destination.write_bytes(image_response.content)
        saved_files.append(str(destination))

    return saved_files

try:
    qq_files = download_qq_stickers()
    print(f'成功下载 {len(qq_files)} 张图片：')
    print(*qq_files, sep='\n')
except requests.RequestException as error:
    print('网络请求失败：', error)

## 任务二 手机归属地查询

为避免在公开仓库中保存完整手机号，请在本机设置环境变量 `LAB_PHONE`，或只在运行前临时修改 `PHONE` 变量。

In [ ]:
PHONE = os.getenv('LAB_PHONE', '').strip()

def save_phone_attribution(phone, output_file='phone_attribution.html'):
    if not re.fullmatch(r'1[3-9]\d{9}', phone):
        raise ValueError('请提供有效的 11 位中国大陆手机号码')

    # 1. 定义请求信息
    url = 'https://m.ip138.com/sj.asp'
    params = {'mobile': phone}

    # 2. 发送请求并获得响应
    response = requests.get(url, headers=HEADERS, params=params, timeout=15)
    response.raise_for_status()
    response.encoding = response.apparent_encoding or response.encoding

    # 3. 将响应源代码存储到本地 HTML
    output_path = Path(output_file)
    output_path.write_bytes(response.content)
    return output_path

if PHONE:
    try:
        html_file = save_phone_attribution(PHONE)
        print('HTML 已保存：', html_file.resolve())
    except (ValueError, requests.RequestException) as error:
        print('查询失败：', error)
else:
    print('未设置 LAB_PHONE，已跳过手机号查询；公开仓库不会保存完整手机号。')

## 本地 JSON 数据检查

In [ ]:
import json

for filename in ('data/companies.json', 'data/devices.json'):
    path = Path(filename)
    with path.open('r', encoding='utf-8') as file:
        records = json.load(file)
    print(f'{filename}: {len(records)} 条记录')